In [0]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [0]:
builder = (SparkSession.builder
           .appName("manage-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [0]:
df = spark.read.format("delta").load("/opt/workspace/data/delta_lake/netflix_titles")
df.show(5)

### Table Constraints

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ALTER COLUMN title DROP NOT NULL

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ADD CONSTRAINT validType CHECK (type IN ('Movie', 'Show','TV Show'));

### Cloning Delta Tables

In [0]:
%%sparksql
CREATE OR REPLACE TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles_shallow_clone` SHALLOW CLONE delta.`/opt/workspace/data/delta_lake/netflix_titles`

### Update Schema

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ADD COLUMNS (ID INT)

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ALTER COLUMN country AFTER show_id;

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ALTER COLUMN release_year AFTER show_id;

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` SET TBLPROPERTIES (
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5',
    'delta.columnMapping.mode' = 'name'
  )

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` RENAME COLUMN listed_in TO genres

In [0]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` DROP COLUMN ID

In [0]:
spark.stop()